# Uncertainty Quantification Head

Fit a two-layer Transformer head to lookback ratios and recent attention weights. The experiment adapts the method to token-level code defect detection. [Paper](https://arxiv.org/abs/2505.08200v1)

Train an independent probe on each benchmark. Line and sample scores use the maximum annotated token score. F1 thresholds come from validation.

In [1]:
from pathlib import Path
import sys

if Path("traces.py").is_file():
    sys.path.insert(0, str(Path.cwd().parent))

In [2]:
import pandas as pd
import torch
from IPython.display import display

from experiments.data import DATASETS, PILOT_LIMITS
from experiments.evaluation import TraceStore, evaluate
from experiments.methods import UncertaintyQuantificationHead

pilot = False
limits = PILOT_LIMITS if pilot else None
torch.set_num_threads(min(4, torch.get_num_threads()))
store = TraceStore(quantize=True)

In [3]:
results = []
for benchmark in DATASETS:
    splits = store.load_splits(benchmark, limits)
    results.extend(
        {"benchmark": benchmark, **row}
        for row in evaluate(UncertaintyQuantificationHead(), splits)
    )

metrics = pd.DataFrame(results).set_index(["benchmark", "level"])
display(metrics.round(4))

n  positives  positive_rate  pr_auc  \
benchmark      level                                            
mbpp           token   3373        660         0.1957  0.3326   
               line     509         54         0.1061  0.3892   
               sample    74         31         0.4189  0.5328   
humaneval      token   3714        665         0.1791  0.2189   
               line     484         55         0.1136  0.2270   
               sample    46         19         0.4130  0.4749   
humaneval_java token   1157        205         0.1772  0.2882   
               line     155         15         0.0968  0.3446   
               sample    36         11         0.3056  0.4484   
defects4j      token   3901       2220         0.5691  0.8962   
               line     448        148         0.3304  0.8180   
               sample    86         35         0.4070  0.5738   

                       average_precision  roc_auc  precision  recall      f1  \
benchmark      level                                                           
mbpp           token              0.3335   0.7072     0.2970  0.6318  0.4041   
               line               0.3948   0.8450     0.2867  0.7593  0.4162   
               sample             0.5411   0.5214     0.4189  1.0000  0.5905   
humaneval      token              0.2196   0.5883     0.2015  0.9850  0.3345   
               line               0.2319   0.6900     0.1349  0.9636  0.2366   
               sample             0.4876   0.5273     0.4186  0.9474  0.5806   
humaneval_java token              0.2910   0.6844     0.2975  0.4049  0.3430   
               line               0.3638   0.8576     0.3000  0.4000  0.3429   
               sample             0.4734   0.6436     0.4000  0.7273  0.5161   
defects4j      token              0.8962   0.8677     0.7752  0.8653  0.8178   
               line               0.8186   0.8798     0.6230  0.8041  0.7021   
               sample             0.5796   0.5630     0.4048  0.9714  0.5714   

                       threshold  
benchmark      level              
mbpp           token      0.2875  
               line       0.2883  
               sample     0.2881  
humaneval      token      0.1446  
               line       0.1448  
               sample     0.1454  
humaneval_java token      0.1132  
               line       0.1136  
               sample     0.1135  
defects4j      token      0.4637  
               line       0.4721  
               sample     0.4606